In [4]:
import pandas as pd

# ==========================
# 1. List your files & months
# ==========================
files = {
    "Chandarana July data.xlsx": "July",
    "Chandarana August data.xlsx": "August",
    "Chandarana September data.xlsx": "September",
    "Chandarana October data.xlsx": "October",
    "Chandarana November data.xlsx": "November",
    "Chandarana December data.xlsx": "December",
    "Chandarana January data.xlsx":"January"
}

sheet_name = "Table1"
dept_col = "Department"
item_col = "Item Desc"

# ==========================
# 2. Department Reclassification Function
# ==========================
def reclassify_department(row):
    dept = str(row[dept_col]).strip().lower()
    item = str(row['clean_item'])

    if dept == "all cleaners":
        if "multipurpose glass" in item:
            return "Glass Cleaners"
        elif any(x in item for x in ["all purpose", "multipurpose", "multi surface"]):
            return "Multipurpose Cleaners"
        elif "kitchen cleaner" in item:
            return "Kitchen Cleaners"
        elif any(kw in item for kw in ["window", "win"]) and any(size in item for size in ["300ml", "320ml"]):
            return "Window Cleaners"
        elif any(x in item for x in ["glass", "window"]):
            return "Glass Cleaners"
        elif "home dry" in item:
            return "Home Dry Cleaners"
        elif any(x in item for x in ["carpet", "upholstery"]):
            return "Carpet Cleaners"
        elif "bathroom" in item:
            return "Bathroom Cleaners"
        elif "floor" in item:
            return "Floor Cleaners"
        else:
            return "All Cleaners"
    elif dept == "antiseptic":
        if any (x in item for x in ["floor","f/cleaner"]):
            return "Floor Cleaners"
        elif "disinfectant" in item:
            return "DISINFECTANT"
    elif dept == "disinfectant":
        if any ( x in item for x in ["floor","f/cleaner"]):
            return "Floor Cleaners"
    
    elif dept == "bleaches":
        if any(x in item for x in ["colour", "color", "colours"]):
            return "Colours Bleach"
        else:
            return "Regular Bleach"

    elif dept == "toilet cleaner":
        if any(x in item for x in ["block", "blocks"]):
            return "Airfreshener Blocks"
        elif any(x in item for x in ["toilet ball", "ball", "balls"]):
            return "Toilet Balls"
        else:
            return "Toilet Cleaners"

    elif dept == "washing up liquid":
        if any(x in item for x in [
            "dishwashing", "dish washing", "d/wash", "dishwash", "dish wash",
            "d/washing", "Dishwanging", "Dishwasher"
        ]):
            return "Dishwashing"
        elif "hw" in item:
            return "HAND WASH"
        else:
            return "Washing Up Liquid"

    elif dept == "fabric conditioner":
        if "toilet cleaner" in item:
            return "Toilet Cleaners"

    elif dept == "medicare":
        if any(x in item for x in ["waterguard","aquaguard"]):
            return "Water Purifiers"

    elif dept == "stationaries":
        if "glue" in item:
            return "Glue"
    elif dept == "women/unisex shower gel":
        if any(x in item for x in ["bath oil", "elysum", "elysium","tabs","crystals","pan","salts","salt"]):
            return "Bath oils"
        

    return row[dept_col]


# ==========================
# 3. Process all files
# ==========================
final_df = pd.DataFrame()

for file, month in files.items():
    print(f"Processing: {file}...")

    df = pd.read_excel(file, sheet_name=sheet_name)

    df["clean_item"] = df[item_col].astype(str).str.lower()

    # Reclassify department
    df["Cleaned Department"] = df.apply(reclassify_department, axis=1)

    # Add month column
    df["Month"] = month

    # Remove helper column
    df.drop(columns=["clean_item"], inplace=True)

    final_df = pd.concat([final_df, df], ignore_index=True)

# ==========================
# 4. Save combined cleaned file
# ==========================
output_path = "Chandarana_Combined.xlsx"
final_df.to_excel(output_path, index=False)

print("✅ ALL FILES CLEANED & DATA COMBINED!")
print("📁 Saved as:", output_path)


Processing: Chandarana July data.xlsx...
Processing: Chandarana August data.xlsx...
Processing: Chandarana September data.xlsx...
Processing: Chandarana October data.xlsx...
Processing: Chandarana November data.xlsx...
Processing: Chandarana December data.xlsx...
Processing: Chandarana January data.xlsx...
✅ ALL FILES CLEANED & DATA COMBINED!
📁 Saved as: Chandarana_Combined.xlsx


In [5]:
import pandas as pd
from rapidfuzz import process, fuzz

# --- Load your dataset ---
df = pd.read_excel("Chandarana_Combined.xlsx")

# --- Clean item descriptions for consistency ---
def clean_text(x):
    if pd.isna(x):
        return ""
    return (
        str(x)
        .upper()
        .replace(".", "")
        .replace(",", "")
        .replace("-", " ")
        .strip()
    )

df["Clean_Item"] = df["Item Desc"].apply(clean_text)

# --- Build reference dataset (only rows with supplier names) ---
ref_df = df[df["VendorName"].notna() & (df["Month"] != "August")][["Clean_Item", "VendorName"]]
ref_dict = dict(zip(ref_df["Clean_Item"], ref_df["VendorName"]))

# --- Prepare to fill missing suppliers for August ---
august_mask = (df["Month"] == "August") & (df["VendorName"].isna())

# --- Function to fuzzy match ---
def find_supplier(item):
    if not item:
        return None
    match = process.extractOne(item, ref_dict.keys(), scorer=fuzz.token_sort_ratio)
    if match and match[1] >= 85:  # match score threshold
        return ref_dict[match[0]]
    return None

# --- Apply fuzzy match only to August rows missing VendorName ---
df.loc[august_mask, "VendorName"] = df.loc[august_mask, "Clean_Item"].apply(find_supplier)

# --- HARPIC FIX AFTER FUZZY ---
df["VendorName"] = df["VendorName"].astype(str)

df.loc[
    (df["VendorName"].str.lower() == "sparkle brands limited") &
    (df["Item Desc"].str.lower().str.contains("harpic")),
    "VendorName"
] = "TOWFIQ  K  LIMITED HQ"

# --- Save result ---
df.drop(columns=["Clean_Item"], inplace=True)
df.to_excel("Suppliers_Filled.xlsx", index=False)

print("✅ Done! Missing supplier names for August have been filled using fuzzy matching.")
print("🛠 HARPIC vendor correction applied.")
print("💾 File saved as 'Suppliers_Filled.xlsx'")


✅ Done! Missing supplier names for August have been filled using fuzzy matching.
🛠 HARPIC vendor correction applied.
💾 File saved as 'Suppliers_Filled.xlsx'


In [6]:
import pandas as pd

df = pd.read_excel("Suppliers_Filled.xlsx")

# Clean branch names
df["Branch"] = (
    df["Branch"].astype(str)
        .str.replace("\u00A0", " ", regex=False)  # non-breaking space
        .str.replace("–", "-", regex=False)       # long dash to normal hyphen
        .str.strip()
)

# DO NOT drop duplicates (to preserve sales)
df.to_excel("final_data.xlsx", index=False)